# 🤖 CIFAR-10 Image Classification with a CNN for the e-puck Robot

**Applied Robotics and AI Project, Part 2 · Assignment 3** · BEng (Hons) Robotics and Artificial Intelligence, University of Hertfordshire

A convolutional neural network (CNN) is trained on the CIFAR-10 dataset **off the robot** (on a Colab GPU). The trained model is then tested on my own photos and on an image from the **e-puck camera in Webots**, and finally made **smaller and faster** so that it can run on the robot without disturbing its movement.

| Step | Content | Assignment requirement |
|:-:|---|---|
| 1 | Setup | |
| 2 | Load the CIFAR-10 dataset | |
| 3 | Build the CNN | **Show all the layers of the CNN** |
| 4 | Train the CNN | |
| 5 | Test accuracy and error analysis | **Show the accuracy score** |
| 6 | Predict my own images and the Webots camera image | **Predictions on own images + e-puck camera image** |
| 7 | Optimise the model for the robot | **Optimisation proposal** |
| 8 | Save and download the results | Model file for Webots, figures for the report |
| 9 | Summary, key findings and future work | |

### ▶ How to run
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all** (about 5–7 minutes on a T4 GPU)
3. At step 6, click **Choose Files** to test your own images, or **Cancel upload** to use the saved images (my car, cat and bird photos and three Webots e-puck camera images: dog, deer and horse).

Every cell prints what it is doing (**▶ start**, **•** progress, **✔ done** with the time taken), so you can follow the run in the output.

---
## Background and motivation

A mobile robot such as the **e-puck** sees the world through a small colour camera (52 × 39 pixels in Webots). In the first part of the project the robot recognised objects only by **colour** (red, green and blue blocks). Colour alone cannot tell a dog from a deer or a car from a truck. A **CNN** learns shapes and textures from thousands of labelled images, so it can recognise *what* an object is.

**CIFAR-10** is a standard benchmark with 10 everyday classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck) and small 32 × 32 images, a similar resolution to the e-puck camera. This makes it a good first step towards object recognition on the robot.

Training needs much more computing power than a small robot has, so the CNN is **trained here on a GPU** and only the trained weights are copied to the robot, which runs the much cheaper forward pass.

---
## 1. Setup
Imports, random seed and the device (GPU if available).

In [ ]:
#@title 1.1 Install packages and import libraries
print("▶ Step 1.1  Installing torchinfo and datasets...")
!pip -q install torchinfo datasets
print("✔ Packages installed")

import copy, io, json, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchinfo import summary
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from PIL import Image

from contextlib import contextmanager

from IPython.display import HTML, display

def header(title):
    """Print a boxed heading for each step, so every run is easy to follow in the output."""
    line = "─" * (len(title) + 6)
    print(f"\n┌{line}┐\n│ ▶  {title}  │\n└{line}┘")

@contextmanager
def step(title):
    """Boxed heading at the start, and the time taken at the end."""
    header(title)
    start = time.time()
    yield
    print(f"✔ Done in {time.time() - start:.1f} s")

def info(message):
    print(f"   • {message}")

def show_table(data, title=None, fmt=None, bar=None, highlight=None, index=False):
    """Show a table in a neat box: title bar, coloured header row, striped rows."""
    df = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    st = df.style
    if not index:
        st = st.hide(axis="index")
    if fmt:
        st = st.format(fmt)
    if bar:
        st = st.bar(subset=bar, color="#8FB8DE", vmin=0, vmax=100)
    if highlight:
        st = st.highlight_min(subset=highlight, color="#DDF0DA")
    numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric:
        st = st.set_properties(subset=numeric, **{"text-align": "right"})
    st = st.set_table_styles([
        {"selector": "", "props": "border-collapse: collapse; margin: 0; background: #FFFFFF; color: #1A1A1A;"},
        {"selector": "th.col_heading", "props": "background: #1F4E79; color: #FFFFFF; font-weight: 600; "
                                                "padding: 8px 14px; text-align: left; border: none;"},
        {"selector": "th.row_heading", "props": "background: #EEF3F9; color: #1F4E79; font-weight: 600; padding: 6px 12px;"},
        {"selector": "th.blank", "props": "background: #1F4E79; border: none;"},
        {"selector": "td", "props": "padding: 6px 14px; border-top: 1px solid #E3E8EF; color: #1A1A1A;"},
        {"selector": "tbody tr", "props": "background: #FFFFFF;"},
        {"selector": "tbody tr:nth-child(even)", "props": "background: #F6F9FC;"},
    ])
    caption = (f'<div style="background:#EEF3F9;color:#1F4E79;font-weight:700;padding:8px 14px;'
               f'border-bottom:2px solid #1F4E79">{title}</div>') if title else ""
    display(HTML(f'<div style="display:inline-block;border:1px solid #C9D3E0;border-radius:8px;overflow:hidden;'
                 f'margin:6px 0 12px;box-shadow:0 1px 4px rgba(0,0,0,.10);font-family:Arial,Helvetica,sans-serif;'
                 f'font-size:13px;background:#FFFFFF">{caption}{st.to_html()}</div>'))

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.family": "serif", "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, GREEN, AMBER, RED = "#1F4E79", "#8FB8DE", "#5FA857", "#F6C453", "#C0392B"

print("✔ Libraries imported")
info(f"PyTorch {torch.__version__}, torchvision {torchvision.__version__}")
info(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda"
                             else "  ⚠ no GPU: choose Runtime → Change runtime type → T4 GPU"))
info(f"Figures and tables are saved in the '{OUT}/' folder")

The model is saved as `cnn_model.py`. The Webots controller imports this same file, so the network trained here is exactly the one that runs on the robot.

In [ ]:
%%writefile cnn_model.py
"""CNN for CIFAR-10 (32 x 32 RGB images, 10 classes).

The same file is used by the training notebook and by the Webots controller,
so the network that is trained is exactly the network that runs on the robot.
"""
import torch
import torch.nn as nn

CLASSES = ("airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")

# Per-channel mean and standard deviation of the CIFAR-10 training set.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


def conv_block(c_in, c_out, dropout):
    """Two 3x3 convolutions (each followed by batch norm and ReLU), then 2x2 max pooling."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),          # halves the height and width
        nn.Dropout(dropout),
    )


class CifarCNN(nn.Module):
    """VGG-style network: three convolution blocks and a two-layer classifier.

    Input 3 x 32 x 32 -> block 1 (32 ch, 16 x 16) -> block 2 (64 ch, 8 x 8)
    -> block 3 (128 ch, 4 x 4) -> fully connected 256 -> 10 class scores.
    """

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32, 0.2),
            conv_block(32, 64, 0.3),
            conv_block(64, 128, 0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def preprocess(rgb_uint8):
    """Turn an H x W x 3 uint8 RGB tensor into a normalised 1 x 3 x 32 x 32 batch.

    The largest centred square is cropped first, so the object is not squashed,
    and then resized to 32 x 32 with antialiasing.
    """
    img = rgb_uint8.permute(2, 0, 1).float().div(255.0)          # 3 x H x W, range 0..1
    _, h, w = img.shape
    s = min(h, w)
    top, left = (h - s) // 2, (w - s) // 2
    img = img[:, top:top + s, left:left + s]
    img = nn.functional.interpolate(img.unsqueeze(0), size=(32, 32), mode="bilinear",
                                    align_corners=False, antialias=True)
    mean = torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(1, 3, 1, 1)
    return (img - mean) / std

In [ ]:
import importlib, cnn_model
importlib.reload(cnn_model)
from cnn_model import CifarCNN, CLASSES, CIFAR_MEAN, CIFAR_STD, preprocess
print("✔ CNN definition loaded from cnn_model.py")
info("Classes: " + ", ".join(CLASSES))

---
## 2. Load CIFAR-10

60,000 colour images of 32 × 32 pixels in 10 classes: **50,000 for training** and **10,000 for testing**.

* **Augmentation** (training only): random crop with 4-pixel padding and horizontal flip, so the model does not just memorise the images.
* **Normalisation**: each colour channel is scaled with the CIFAR-10 mean and standard deviation.

The data is loaded from the Hugging Face mirror (fast). If that fails, it falls back to the official site.

**Speed:** all 60,000 images are stored once in GPU memory (about 180 MB), and the random crop and flip are done on the GPU for a whole batch at once. Colab has only 2 CPU cores, so preparing images on the CPU would make the GPU wait; this way one epoch takes a few seconds instead of about 20.

In [ ]:
class ImageList:
    """The original PIL images and labels (used for showing pictures)."""
    def __init__(self, images, labels):
        self.images, self.targets = images, list(labels)
    def __len__(self):
        return len(self.images)

MEAN_T = torch.tensor(CIFAR_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_T = torch.tensor(CIFAR_STD, device=DEVICE).view(1, 3, 1, 1)

def to_tensor(images):
    """PIL images -> one uint8 tensor (N, 3, 32, 32) on the GPU."""
    arr = np.stack([np.asarray(im, dtype=np.uint8) for im in images])
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous().to(DEVICE)

def random_crop_flip(x):
    """Random 32x32 crop from a 4-pixel zero-padded image and random horizontal flip, for a whole batch on the GPU."""
    n, c = x.shape[:2]
    xp = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n,), device=x.device)
    j = torch.randint(0, 9, (n,), device=x.device)
    ar = torch.arange(32, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None]
    cols = (j[:, None] + ar)[:, None, None, :]
    out = xp[torch.arange(n, device=x.device)[:, None, None, None], torch.arange(c, device=x.device)[None, :, None, None], rows, cols]
    flip = torch.rand(n, device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], out.flip(3), out)

class GPULoader:
    """Batches straight from GPU memory: normalised, with optional augmentation."""
    def __init__(self, x_uint8, y, batch_size, shuffle=False, augment=False):
        self.x, self.y, self.bs, self.shuffle, self.augment = x_uint8, y, batch_size, shuffle, augment
    def __len__(self):
        return (len(self.y) + self.bs - 1) // self.bs
    def __iter__(self):
        n = len(self.y)
        order = torch.randperm(n, device=self.x.device) if self.shuffle else torch.arange(n, device=self.x.device)
        for k in range(0, n, self.bs):
            idx = order[k:k + self.bs]
            xb = self.x[idx].float().div_(255)
            if self.augment:
                xb = random_crop_flip(xb)
            yield (xb - MEAN_T) / STD_T, self.y[idx]

from tqdm.auto import tqdm

def load_cifar10():
    try:                                                  # fast mirror
        info("Downloading from the Hugging Face mirror (uoft-cs/cifar10)...")
        from datasets import load_dataset
        ds = load_dataset("uoft-cs/cifar10")
        sets = {}
        for split in ("train", "test"):
            imgs = [im.convert("RGB") for im in tqdm(ds[split]["img"], desc=f"   preparing {split} images", unit="img")]
            sets[split] = (imgs, ds[split]["label"])
    except Exception as err:                              # official site (slower)
        info(f"Mirror not available ({type(err).__name__}); downloading from the official CIFAR-10 site...")
        sets = {}
        for split, train in (("train", True), ("test", False)):
            raw = torchvision.datasets.CIFAR10("data", train=train, download=True)
            sets[split] = ([raw[i][0] for i in tqdm(range(len(raw)), desc=f"   preparing {split} images")], raw.targets)
    return sets

with step("Step 2.1  Loading CIFAR-10"):
    sets = load_cifar10()
    train_set = ImageList(*sets["train"])
    test_set = ImageList(*sets["test"])
    info(f"Copying all images to {DEVICE.type.upper()} memory...")
    X_train, y_train = to_tensor(train_set.images), torch.tensor(train_set.targets, device=DEVICE)
    X_test, y_test = to_tensor(test_set.images), torch.tensor(test_set.targets, device=DEVICE)
    train_loader = GPULoader(X_train, y_train, 128, shuffle=True, augment=True)
    test_loader = GPULoader(X_test, y_test, 1000)
    info(f"{len(train_set):,} training images in {len(train_loader)} batches of 128 (augmented on the {DEVICE.type.upper()})")
    info(f"{len(test_set):,} test images in batches of 1,000")

show_table(pd.DataFrame({"Split": ["Training", "Test"], "Images": [len(train_set), len(test_set)],
                         "Size": ["32 × 32 × 3"] * 2, "Classes": [10, 10]}),
           "CIFAR-10 dataset", fmt={"Images": "{:,}"})

In [ ]:
header("Step 2.2  Showing two example test images per class")
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for c, name in enumerate(CLASSES):
    idx = [i for i, y in enumerate(test_set.targets) if y == c][:2]
    for r in range(2):
        axes[r, c].imshow(test_set.images[idx[r]]); axes[r, c].axis("off")
    axes[0, c].set_title(name, fontsize=9)
plt.suptitle("CIFAR-10: two test images per class", y=1.02)
plt.savefig(f"{OUT}/cifar10_samples.png", bbox_inches="tight"); plt.show()
print(f"✔ Saved {OUT}/cifar10_samples.png")

---
## 3. Build the CNN: all layers

A VGG-style network: **three convolution blocks**, then a **classifier**.

| Stage | Layers | Output size |
|---|---|---|
| Input | RGB image | 3 × 32 × 32 |
| Block 1 | 2 × (Conv 3×3, 32 filters → BatchNorm → ReLU) → MaxPool → Dropout 0.2 | 32 × 16 × 16 |
| Block 2 | 2 × (Conv 3×3, 64 filters → BatchNorm → ReLU) → MaxPool → Dropout 0.3 | 64 × 8 × 8 |
| Block 3 | 2 × (Conv 3×3, 128 filters → BatchNorm → ReLU) → MaxPool → Dropout 0.4 | 128 × 4 × 4 |
| Classifier | Flatten → Dense 256 → BatchNorm → ReLU → Dropout 0.5 → Dense 10 | 10 class scores |

In [ ]:
header("Step 3.1  Building the CNN and printing its summary")
model = CifarCNN().to(DEVICE)
print(summary(model, input_size=(1, 3, 32, 32), col_names=("output_size", "num_params"),
              depth=3, device=DEVICE, verbose=0))
print("✔ CNN built on", DEVICE)

In [ ]:
header("Step 3.2  Passing a test input through every layer to list its output shape")
rows, x = [], torch.zeros(1, 3, 32, 32, device=DEVICE)
model.eval()
with torch.no_grad():
    for name, layer in model.named_modules():
        if len(list(layer.children())) == 0:
            x = layer(x)
            rows.append({"Layer": name, "Type": layer.__class__.__name__,
                         "Output shape": " × ".join(map(str, x.shape[1:])),
                         "Parameters": sum(p.numel() for p in layer.parameters())})
layers_df = pd.DataFrame(rows); layers_df.index += 1
TOTAL_PARAMS = int(layers_df["Parameters"].sum())
layers_df.to_csv(f"{OUT}/model_layers.csv")
show_table(layers_df, "All layers of the CNN", fmt={"Parameters": "{:,}"}, index=True)
print(f"✔ {len(layers_df)} layers, {TOTAL_PARAMS:,} parameters ({TOTAL_PARAMS * 4 / 1e6:.2f} MB as 32-bit numbers)")
info(f"Saved {OUT}/model_layers.csv")

---
## 4. Train

| Setting | Value |
|---|---|
| Optimiser | SGD, momentum 0.9 (Nesterov), weight decay 5 × 10⁻⁴ |
| Learning rate | One-cycle schedule, peak 0.1 |
| Loss | Cross-entropy with label smoothing 0.1 |
| Epochs / batch size | 30 / 128 (mixed precision on the GPU) |

Each epoch shows a progress bar (391 steps of 128 images) with the training accuracy and loss, then the test (validation) accuracy and loss. ★ marks the best epoch so far. With the data on the GPU, an epoch takes only a few seconds on a T4, so all 30 epochs finish in a few minutes. Use the `EPOCHS` slider for a quicker test run (fewer epochs give a lower accuracy).

In [ ]:
EPOCHS = 30  #@param {type:"slider", min:5, max:40, step:5}
torch.backends.cudnn.benchmark = True        # pick the fastest convolution algorithms for this GPU
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.1, epochs=EPOCHS, steps_per_epoch=len(train_loader))
scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

def evaluate(net, loader=test_loader, device=DEVICE):
    """Return (loss, accuracy) on a data loader."""
    net.eval(); correct = total = 0; loss_sum = 0.0
    with torch.inference_mode():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = net(xb)
            loss_sum += F.cross_entropy(out, yb, reduction="sum").item()
            correct += (out.argmax(1) == yb).sum().item(); total += yb.numel()
    return loss_sum / total, correct / total

def train_epoch(net, opt, sched=None, bar=None):
    """One pass over the training set. Returns (loss, accuracy). Updates the progress bar if given."""
    net.train(); total = 0
    loss_sum = torch.zeros((), device=DEVICE); correct = torch.zeros((), device=DEVICE)   # kept on the GPU (no waiting)
    for step, (xb, yb) in enumerate(train_loader, 1):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            out = net(xb); loss = criterion(out, yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        if sched: sched.step()
        loss_sum += loss.detach().float() * yb.numel()
        correct += (out.argmax(1) == yb).sum(); total += yb.numel()
        if bar is not None:
            bar.update(1)
            if step % 50 == 0 or step == len(train_loader):
                bar.set_description_str(f"accuracy: {correct.item()/total:.4f} - loss: {loss_sum.item()/total:.4f}")
    return loss_sum.item() / total, correct.item() / total

header(f"Step 4.1  Training for {EPOCHS} epochs on {DEVICE} ({len(train_loader)} steps per epoch)")
info("The best model (highest test accuracy) is saved to cifar10_cnn.pth")
history, best_acc, t0 = [], 0.0, time.time()
for epoch in range(1, EPOCHS + 1):
    bar = tqdm(total=len(train_loader), desc="training...", unit="step",
               bar_format=f"Epoch {epoch}/{EPOCHS}  {{n_fmt}}/{{total_fmt}} {{bar}} {{elapsed}} - {{desc}}")
    tl, ta = train_epoch(model, optimizer, scheduler, bar)
    vl, va = evaluate(model)
    history.append({"epoch": epoch, "train_loss": tl, "train_acc": ta, "test_loss": vl, "test_acc": va})
    best = va > best_acc
    if best:
        best_acc = va; torch.save(model.state_dict(), "cifar10_cnn.pth")
    bar.set_description_str(f"accuracy: {ta:.4f} - loss: {tl:.4f} - val_accuracy: {va:.4f} - val_loss: {vl:.4f}"
                            + ("  ★ best" if best else ""))
    bar.close()
TRAIN_TIME = time.time() - t0
model.load_state_dict(torch.load("cifar10_cnn.pth", map_location=DEVICE))
print(f"✔ Training finished in {TRAIN_TIME/60:.1f} min. Best test accuracy: {best_acc*100:.2f}%")

In [ ]:
header("Step 4.2  Plotting the training curves")
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(h.epoch, h.train_loss, color=BLUE, lw=2, label="training")
ax[0].plot(h.epoch, h.test_loss, color=NAVY, lw=2, label="test")
ax[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
ax[1].plot(h.epoch, h.train_acc * 100, color=BLUE, lw=2, label="training")
ax[1].plot(h.epoch, h.test_acc * 100, color=NAVY, lw=2, label="test")
ax[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title="Accuracy")
for a in ax: a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT}/training_curves.png", bbox_inches="tight"); plt.show()
gap = (h.train_acc.iloc[-1] - h.test_acc.iloc[-1]) * 100
print(f"✔ Saved {OUT}/training_curves.png")
info(f"Final training accuracy {h.train_acc.iloc[-1]*100:.1f}%, test accuracy {h.test_acc.iloc[-1]*100:.1f}% "
     f"(gap {gap:.1f} points: {'small, so little overfitting' if gap < 8 else 'some overfitting'})")

---
## 5. Test accuracy

The trained model is tested on the **10,000 test images** it has never seen.

In [ ]:
header(f"Step 5.1  Classifying the {len(test_set):,} test images")
model.eval(); preds, labels = [], []
with torch.inference_mode():
    for xb, yb in test_loader:
        preds.append(model(xb).argmax(1).cpu()); labels.append(yb.cpu())
preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
TEST_ACC = float((preds == labels).mean())
cm = confusion_matrix(labels, preds)
per_class = cm.diagonal() / cm.sum(1)

print(f"✔ Test accuracy: {TEST_ACC*100:.2f}%   ({(preds == labels).sum():,} of {len(labels):,} images correct)")
show_table(pd.DataFrame({"Class": CLASSES, "Accuracy (%)": (per_class * 100).round(1)})
           .sort_values("Accuracy (%)", ascending=False),
           "Test accuracy per class", fmt={"Accuracy (%)": "{:.1f}"}, bar=["Accuracy (%)"])

In [ ]:
header("Step 5.2  Confusion matrix: which classes are mixed up")
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
ax.set_title(f"Confusion matrix (test accuracy {TEST_ACC*100:.1f}%)")
plt.tight_layout(); plt.savefig(f"{OUT}/confusion_matrix.png", bbox_inches="tight"); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"✔ Saved {OUT}/confusion_matrix.png")
info(f"Easiest class: {CLASSES[per_class.argmax()]} ({per_class.max()*100:.1f}%)")
info(f"Hardest class: {CLASSES[per_class.argmin()]} ({per_class.min()*100:.1f}%)")
info(f"Most common mistake: {CLASSES[i]} predicted as {CLASSES[j]} ({off[i, j]} images)")

In [ ]:
header("Step 5.3  Examples of test images the model got wrong")
wrong = np.where(preds != labels)[0][:12]
fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for a, k in zip(axes.flat, wrong):
    a.imshow(test_set.images[k]); a.axis("off")
    a.set_title(f"true: {CLASSES[labels[k]]}\npred: {CLASSES[preds[k]]}", fontsize=8, color=RED)
plt.suptitle("Examples of mistakes"); plt.tight_layout()
plt.savefig(f"{OUT}/misclassified.png", bbox_inches="tight"); plt.show()
print(f"✔ Saved {OUT}/misclassified.png")

**Comparison with other models on CIFAR-10**

In [ ]:
bench = pd.DataFrame([
    {"Model": "Random guess", "Test accuracy (%)": "10", "Parameters": "–"},
    {"Model": "Simple CNN (3 conv layers, Keras tutorial style)", "Test accuracy (%)": "≈ 70", "Parameters": "≈ 0.12 M"},
    {"Model": "This model (6 conv layers + BatchNorm + Dropout)", "Test accuracy (%)": f"{TEST_ACC*100:.1f}", "Parameters": f"{TOTAL_PARAMS/1e6:.2f} M"},
    {"Model": "ResNet-18 (He et al., 2016)", "Test accuracy (%)": "≈ 93–95", "Parameters": "11.2 M"},
])
show_table(bench, "Comparison with other models on CIFAR-10")

**Why some classes are harder.** Cat, dog, bird and deer are animals with similar shapes, colours and backgrounds, and at 32 × 32 pixels the small details that separate them (ears, snout, legs) are only a few pixels wide. Vehicles (automobile, truck, ship) have strong straight edges and typical backgrounds (road, sea), so they are easier. A deeper network such as ResNet-18 reaches about 94% but has 14 times more parameters, which is too heavy for a small robot.

---
## 6. Predict your own images

When the cell runs, a **Choose Files** button appears under it.

* Pick your images: for example **two of your own photos** and **one image from the e-puck camera** in Webots (`controllers/Assignment_3/captures/`).
* Press **Cancel upload** (or set `IMAGE_SOURCE` to *my saved images*) to use my saved images: **car, cat and bird photos** and three **Webots e-puck camera images** (dog, deer and horse) from the arena. They are stored inside this notebook.
* Run this cell again to try more images; no retraining is needed.

Each image is cropped to a centred square and resized to 32 × 32, the same way the Webots controller does it.

In [ ]:
#@title 6.1 Saved test images (embedded in the notebook, so no download is needed)
SAVED_IMAGES = {
"car.jpg": "...",
"cat.jpg": "...",
"bird.jpg": "...",
"webots_dog.png": "iVBORw0KGgoAAAANSUhEUgAAADQAAAAnCAYAAABAOmveAAARfElEQVR4XmP0Tez/zwAC/yDU//9AmhHEhvEZGP79h7DB6sDsfwyMUKH//+EMBgZGBqgSmPp/QD4T0IB/cPH/GGoQelDMBKuDmQ3VBFLKiOQWkDDYqYwMjIwQNSxMTBDGf6AAwsC/DP+BrgM5lgnoHkagKyAO/w8WZ2BgYvgPN5iRgQkYGH+B5jBCA4URpJcRZAIzxCMgQxgg/mX6Dwmsf0D7wHYA9TDC3AulGRn+ge0BmcPwHxQYQD1AMxj/g+wHGQRUCA54iJngQGIEOhSoj4WZGWgpUPIfyDEgdQxQh4Aj6i8DMmAEaoLHCMhScGABLWECewHoTySLYAEEtpgRYjPIDmgAgqwHm8UMDVCQRSAHgz0MlAXZD3YPE0QvI8Qj4EgByTHBfQ9LGODUwAIJTWAoM4FCBaSXkQFkEVg52NdAdSAPw2Ma6oD/TAyQkAUGADA2QO4AK2GEORDoUZA//oPDmQEctszQVAkMdJBaRiZoAICsADqLCWzOf5RABCV3iF+A7oLKgG2AeRiqHBbLwCQHTD5Awf/gJALR8R8chyCVLFDPAR39D+JhSECCncMAiS1QADCAPfcfmr+YQGEGNOMfIyLvgAMOavl/ZhgDFOoQNhOYyQhPyvCIBUozQawDBwosr/z/D0n+oDCHBSbIUywgghGcH6AeAyU5cLwywg0AeZCRGRJTYC//h4RskOJ1hs9fvjLwMvxhkJASZ5h+SQmY5EHJ5R8kk0Jt+wcLWUYGeBiD/c4ICxhEgIACA+p+BqADIAEFDmxgKkKKO0aYov8gvf/B9oGjgImZEZIBYYH2H8KA+BoUdSAdwND4D0nLoHAJk7/KwMbOyvDi1WeG+ZvPMWS6aTPMW3eE4dnHYwzSJrEM4NAFmsoMKjygxQgD2CiI2YyIlMbwH27ff3hJBXM3rESE6AWVZP+RAhmRtf4zMIG1MIELBWgJBDEX5HBYMELYsNINbglQ4Yx1xxhUpQUYLtx5weBjps4wZcsFhv+s7AycnNwMzOCYZAK5gQHkIFAJCA0riOOhIYsoXCBJlgGcAECBxwT3JFKEMiADaAKCCzFBMxAjKF/HFs76zwCNCAZ4qoLF0j8GZI8g2AwMl7e3MXBxsDH8/fOXwVJbnuH7PyaGm8++gZVIG8cwQFM4A8LhDAwQixgYEHURIzhZwc1lgCXB/3B7wY5HuIIBWp6CXApnw9SAsg6w2IbmHaSoh9n4DxiVsLrlP7yygJhu4FMFtv7uvj5gvvkDVMnCYK8vwWBiqMEgIvqHAVSXgJLA/39/GZiBpRkTMxPDr0/vgEU7KwMztwDDn7//GCbv5wAb9p8Bnssg/P9MDBgA7EdIPQQudBgZkIprqJeAGQuLh/4xQCo0YAUPyoz/QRX9f9SYghYiII8rSYsxaCiIMwgI8TFwcHMzfPv8meHZj59AD/5lYGRiZvj06TPDnz9/GP7+/Mnw58cPBnYBUQZmjq/AYoSVwVTgM8OmA5cYvEwlGS79t4d6BhJLkHwGTZ8MMPvBtRcDtDplgFQySKUsUB0LCzM0vcNjCFIPgbjM0DBgYvoPTy4wwyABBsyOOvEM0hIXGFiBRfGzV28ZXr7/wfDs3XcGeQk+hq/ffzI8efmR4fO3nwy/gJ4R4mZn4BP4ysDBycnwm4GD4fL9t2AbZq47zMDLeYKBg40dmAdZGHRlBBi+q8UxQEoXRkSeYkSuIqAxBM5ojPBqgwWUFEBBA29WgFLnf2idAw2Y/0glHyzRIwoRBgYuXl6GD69eMtx99oHhzJ1PQCV/GG4+es3w7+8fcDvu//+/4Lz25v0XBsZnHxnERfiAhQgPUO4vOGk6RjQxIMro/0DPMjCwgN3xH5pP/oE9DmmpMMDTGjzvMECrGKB7WdhYYaUMUgn3nxlSSv2HRTXMgyA1kGiHSEHYtx+8YGD5+43h8auvwGTGxADyx59fPyAeAgXOv3/g0P716w8DM8s/hreffzKwcrCCI4AB6FkWVgYGeCEMazdCiziQl5jA3mMAe/r/fyQPwYo7aKXEBLSLBVTMMgNrzf/Qouc/NGTA2v/B4oMZahhQ9j8oiSJaAKB8dIvZgUHowRKG3z+/M/z9xcDwF2jwr58/GEAFAgiDXP73739gXgLGCDA/sbCzMzD9+QH0Awu4QGFjgRRMDCgA2gRjhCY5RkTeAoU2qMUAcjILIyvDl9fXGN49vsbw99tXBpYrB5dBK0KI401cYhigYQEuX/+jVLRM4G7Df7CnYAUz2K8M31TjGRS+zmR49vYdwy9gm/bjpy8MoHgGV3NAR/36DfQQMOaZ/v9iYGT5wiAnysvw9N0PcP5gYwUFEiOs0GYAJVFQ8mICBQLTH0gzAmja3y+fgYEFKXD+fP/A8PndbYaPX7nAsf+fEZiEuXkYWBhgDoaGzpndi5HSKCxUGMHxDG5HwYtLWEZkgNbwjAweCgIMR2+8BhYGvxnU9azAevRs/RggvQpI8r1wdBc45Nh+32L49+c3AyMzC8O1kzuR4gbSIoAIQNMdw39EEc0IkockdUYGTgZIkf+fAeYPlv/wIpERyYz/0FiCJWRwQoSE4H9GpOKFEam58p9hxwMWBhZgSfX/+19ItAED69Kh9WCz/jFzgmMDlHJAdr5kUWaQF3/E8PgdMI3+R1TgIJf/R+4jMSCXZiBjmeDuhHRGIYHNAC3tWKAuZ4CkJQYGpNqKAaUaZ0CEFihSGZH8ilznMjGzMnBzcTPAm0ywFPDvD9hwmDiormNhZWFgYgJVwqj9LgbksIUFODiMGaHuhAYAJMMzIANITwpSpjHA+9VI/oJ6HFzOQ+xhhKZpBrij/4NKMjD+y8DDASyTGIGOB/P/QwPlPzhFgNXBWwX/GV79F2dQk+IDqvkHxqDWBaSpDO7LwvUywNwH0guKTWDJCOnJ/oXrZWCAmAFpQsMCHzkW4FEHMRecNKF+YUQKElCI/4M6HsSWFOJm+PnrF8O//5CiGtxvAccSiA8ai0B4siBAE1jp/kJNCf+h+eH/f7QUwgDl/4M6CGI+A7xqYQDXWSwp5toMoDTLBB48AIYuCwskrwDTEai3+A/Y5vr7D1Lk/gPWJ7//QJIHqFEKGo8AFRRsrMwMwCYHg6K+GENR51KG3swkBnZOdgZWNjZgvQNKVkzgvAbS/wdYEPz59ZvhF8t7hicv3jE8efORIcvDlOE3MBB+AZtHf4HFOqipxAwJagZWoNmgyp8V6C5moCAomYKGDWB2//2L5p6D07r+//79B2jgbwYY/Q3YVHn/8TuwSPzJAHKsMD8ng6AgL4OgsAADDz8fAycPN7iJAulCMzCg5zVwnoIQ4LgEM/9DSkxwVoBVtpAKBexZRqSM+BdYM//9/RvYLvzE8PnDB4YPwKrg3dv3DG/efmL49vMPgwAvB4OYKD8DH9AtbGyswL4ZGzBAWRjYODgYGKvC4kG9CAbkjA0rGSD5EDmBMaAAUHpnBXYQOYChyMnBwsDDzQHsE7Ex8PJxM3BwAPtH3FzAdhsHAyuwIgXFzs/v3xlA+NvXbwxfvwDx1+8M34DtvW/ffzN8B7YifgDxr9/AEad/iOQKDizkAAAFAthh0CQH5oOaRpDkzPKPkYXhL7is/M8AGoBhYkAU2SAWO9jBTAzswMqPC9hcYWdnATqWjYGNjQXsaFDSYgM6mB3Y4GQDeoIVGErMwOQBCiFIgcIITtsgK7gF+CExBrTjPyj5ApPZj28/GH6APApsvP4AevQ70KM/vv9g+A70KMiDP3/+Anr6FwOo2fTzzz94NQHSD/Hsf2gz7R8437JY6kozgNIlCysrMM2zMoBokGNBNAswD4Bo8FAXI7TOYWJC1D2wIpABqbRnhGROkGWIWIcmN9hgJjiRMjEwA3u53HxsDNx8fPBkCykLYK17UAEBaX/9ASbBP79/AZtXv8DdEVB+A+c7YECA2KDW/G+gGhY1fW1oMvrPAC0N4GkalEcYQYUFEzNaUoNyIZkD0vQAF07/4GaBy6j/iCIeVBoh5xNIQgAV58CQ/f0Hoo8Roh5cV/3/B2+BgDuJwABnBgYwBw9Ezb8/kJY8AyiAQSkCWDiAWh4sLMBkglKb/oeECqT0BDYswKXaHwZIQ54B4XlQrIAcAK9l/8N8Ca6nIJkf6GBQfgBhRkZEYgYNWIKHxSCBCM+l/yAtDEgTC9oiABX/IL3gUh/oDlBTHqifEVjiwcb1/gFLR3ASB3qO5fe3b9AQBgcxPKQYYUkLSkMqPFjaQhQUkAFXiN7/f/+CqwAGJmgL+R9SqxwaABBvAzuG4GoCmr+YYD1RZmi2AKr6C9X7D0FDWu7A8AIp+/cfPkoMLSwZQLEF7OCxgEMUHNjw0gWSbpEH9UA+hdRhUCdBK0CQx/+DDYfGEHSIFqyXhRlsNhPYM8DiBhi64MzMCG2gMEDzCjhv/WdAKWpBmv5BR2VBNChGQJUT0F5wYDNBRodgDWZwDAHNZQE1R0DhBKpAIXmcCdwCBqV5sBP/QWpmkKP/AZPEv1+gFjJwxI2ZGcUBoNAD5zdg6xlsDqzpA0paIAwqVECxDR02Aw2lwVoR///+AZeEoAoU0s9hhMQUeKgYOBAAGtsA5hMmkJ0gjwArZ7AH/kPG8hiB7UdQ7IB6liyQkIeELqhC+w+OakZ4awE5g4IczMLBgcj4oJQGSvfQ2AKZwvjvD1wvrBPGwMQCTtbg5hA4D0BKQkZgV5UR5HHoQCFowgBSEIGLSgbwmAa4oIDExt/fv8AeBXmciZWNgQHoMVDhAErqjNAkzAJqhsBKIFAIMLGzMsA8wcAIbX/8h2ZucFb5Cx4LAPkKns+AJRA4lMDq/oB7qqAmyf8/IPY/YPHMCvQTEANLS1AF+x/kEJABwJhjAve/IVUCMxuomcQCnsUAqwEV1T9+gMssFi5g542dA5IqQIECxKAUA25+gmIfWvexsLCzgnM6KG2Dov4fsCIDKQI7kAEyOAHyMGQeCeQjoOUs7AwM4JLqD7gQ+A8MuX/A7i8DuDQDJkdQGw6UPEA9UaAYqDj9AyyaYaHIxMYB9CAL2KOgWATFKgOo2AXG9l+gB0AxCU62QM+y8PKB3ccINA+crME+AJa+v0A9X2AAAGMKFBF/fwFHloAVMwsk/UIKbkZQJgaFNqzUgZS94CT2/y8k5BlAjdU/34C9zX9gTzGDDAS2p5g5gTELcvw/SN3y5+dvBiYmyJQLKHmAqyxQSDKDYguY/0D55vdP8EAKqN0GjnFQ/gO2OpghvTyG/5CSCtLS+AspqMCxApJnAuVjUEP6H3CI7Ce4xAN5ngXcTAEb8BcSnaBMB8oX4MwMDIm/vxkgJRMozJiAJTIzcKCQBzgaCsms4GQFTEKQ8gqR95iAAx+gPMLExAypIIF9GEZQyxg4OPLv1z/wNCckzwDNBHqCEaoOnNxhdQ0sqYMDGGg/UA0zMMBBrgcFMGhk6T9wgIcJ2OIA+QOUnyA9VnBBDsl4oLQLymTgUg5cRAJDgoUN4tl/0HoGNHsHTN+g/ABqAIKzGhMrAyRPAWMFXDr9gwQEMEBApc8/cJoHRjkwFYCSMxMwNpjA5v1l+Pf9KwOwyc8Ain0mUHIFtgtBuf8/EyhJsYJLcyZobIGS1h9gRfofOIDCCE664EoJODILHDb78ws4nAYqAqHlFiiU4B0mRshMDXhYGJxe/kPSJROkgACnZ2AaBycn2PQgrNUAax0DYxk0pAUqUv8zs4Nmn8HJEFQ6gRwGCTzguDgo5IF5hBHomf9AM/+Dkh5IDNLKAZdsoBIOXAoDS0QmLj5wP4sBnMRBg5h/GP79+MYAyufgUR9IPQQqvX6BYwIUSiBTQHUTeO4VWBSDPMsEqguY2KDNMEgNDk5koPTNyAAZWARlblA2BrX/QEkXlJSAFSIzyJGggAElT1DqBTqeAdR/gTrqPygAQYUAEyQgwcOEwAD6Cy4sgIEJyntANaCGMsgt/4Fu+gP0BKgzCHIQqJUP6mQCAKQjcc5HU42+AAAAAElFTkSuQmCC",
"webots_deer.png": "iVBORw0KGgoAAAANSUhEUgAAADQAAAAnCAYAAABAOmveAAATEklEQVR4XmN0CW38z8Dwn+H///8MjEAM5DAwAiGIzwAFQFmEGCMDBpsBH2D8B5Zl/M/I8J/xP0gzWD9IENkOEP/SofkMuhqKDIz//jBwcnAwfBewwTSZEeQ6bMJAhwHtYAFJ/2UAuRJmGdh7QIIJbDnQpwxMQENAljOB1TFAHfgPxAOzIYEA0gfVAg8JkAwzxBOMDHD1MGmgsQyIQGNgUJYRZhDg52X4/vkTA8O/3wzC308yvOO0YmDAopcBpBkUAUwQDzJCHcHyn5mJgek/OLwY/iM7hAnkY2CsgTwDEofSwHCGxBAjM2YwAU1Q+H8CaBdQL1DVvf9mwFhhZGBCVgmS+g+KNYgaBqitIGcJ8vMxMDP8Y/j24wfDjXuPGdTkpRgYmHcxcMh7IkIL7BZooDIg0YwQNgszE8hhkCQH9hbQ10xMTKCIAYcMMBYZWIA+/Qe1+D8sWSLFmtCnPcAkwsbAzMLCwMzKzsAM1v+PQen3KaDnQGYzMjxksmCAxSQDNDD+gxM7JIT/Pt7MICwhwfD1yycGdsUABgNFiAMhyRKaghggvoFGDth9oNQDS0GgWGNhYgZ6CJS+waEGiSNw8gLGENgwqG4mBhzg2VaGrwx/GUCBAPQOAyg6GIEe+vfvL8PP378ZWFmYwTGmzHyWgQk56QDN/fcPElMg+ruEMMPvPz8Y/v39wwDSD7MNZO9/qEeQXcDECPMwMPLAkQ0RYGECxtB/cPZBaP0PTXugpAjOyEC94EIDagg4+oAOenR6HsPnzx8ZpMVEgB5iZOAAhTgwcP79/cvw+/cvhr9//jD8+fOPgZWVBZj0mBiAyQGcj0CeBxctTCxgz/4HBggzI1A9MLn9/gMMHHAgg9MLwiuwDAdyKyPUE0BZSHZhhPuVBZw8gFyQ+n///jNgAmhC+Q81Ceo5kLpvP34y/GfmZbj39A0DKzMjA9v3Hwy8vLzgJMcCDJ/vP3+CTAZ6hJmBjY0VHFsgz8E8BKYZmYCeBnr87z9gDP1hePr2K4OsAjDJw4oQqJPgLgP6AJL7oCQo1hkZ4PmfhQUUetAoYWb6j1QwoHoNEkOgIgHh6fcfPzHYB9YynNnZx3D3yWsGBSlhoKO/M7CwsjL8Y2YCVwW/fgMdy/QbyP7H8OsXI9hzzMBkyAyMBVZgnvsHTe7/oG6QM42HeIUR4mBI6gGFC8JeRpTSlhHiZkaIPDCGgAL/GeHlLazOgXkHyYtoPmRgcA5rAAcNI9Dyv0DZj5+/MvwH5h0+Xh5gncYMDPF/DO8+fmHgYGNm+AXMTyCHMIKKWWCsgDwDii1QocTOxgbOO1+//wJ7mAGcYiBu+g9LNchlPAOkLoRFIsjr/6EcFiAAh95/UIX1Hz1WQHxQ+QaRwF6lMTBY+5Ux7FtVz/Du038GHi4OcOb/8u07w4t334Cx8gPoIWBxAUySoCTNBkwRII+wAGOImRmUr1gYhAX4GP4CCwMOjSgGBmhMwZzCCC2N/kNTPiOyE6GeBHsIqg/kH6AhoNIflttgjgeWTv//MrAwfmFg+HyPgZnxL8M/bh2GnwzswIwMTPfAFgAjOPSASYsRFD8gmgXoeDaGHz9/AfFvhjfv34Nj/vM3iCsgeQfuZgZQMlOUFmMQ4P0L1sMPTIqw1gM8y0KT0v9/jJC6lBFSr8HVQX0I9jeQzQJKz6DYYoZnvv/gipP1yz0G1i9XGTiAIc7MwcDwB5hpP77cyiDMzsvw48dHYBL6z8DOwcIAbKUA1XAyqMpLMvz49ZeBl4eL4fW79wyazgUMSkx/GFj/czDsXFbLACqaYUH1H1iaMYCqCaDAxy/fGcSF/jB8+fmfQZgFVm0jMjksRmAxBqIhkQYh/8MVQBoBLMCKCFjug+oDoAImYEj/Z2Pgf7eJ4fX7DwxiMrIMbCz/GPiAzRFgZDFwsDIBK76fDIysXAzszH8Zfv34zCCvqAgM3R8MYqLCDD+/fWX4CSzZlJXFGP7dW8zAxcnBwMjCx+Dh7gws1vkZ/gtpA/WyA5PXf4adSxqA+eY3w+fvfyD1lF06A3LSgecjBkSlD4lniA8geR05/UE8yML+HljhAdPyHyEtoIFsDHJsJxg+83IxaIkKMrCysTO8ffuG4fPHvwycbBzgovfM7TsMnJxcwHTPzCAoIcbw6sULhmt33zAICgkC5dkYPgKLcjZmdgYBYWCbDJjJmRl/M/Bx/Gb4/v0+A/u7Fwxf//EzfPr6C5jPmBgU5KQYdGyjGN5e2QQsHSEOglc34CiBxAskzcAczwitBmFySN4EJTnGP88Zfv8AxsynB6BqjeERsMhlZ+dkEOLiYvj54xsDByc3wy9gqDMx/2N4//41g7QMB4OMpBzDrfu3Gd4/uM3wFxijoIzy8NETBkUZcQY5OS5gLAAtA7YWuIEx9BuYJrm52YApjJNBWEyY4cfj5wwS3P/A+VAR6KFfr24AAwgYk78/MLBwiTCAkiOo1Q+qiYAJguHnf2B+BbbwWP7/AXoE2O4EFi6MwBYGJKmBkuhfhr/AvAyqtH++e8rAcvsdP4MEy2Ng7AArP3ZmBlYOLgY2oE/fffjM8PrNW6An/zAI8nEz3Hn4mOETML3/BdbkP4BFMAPjH2Cm/g502G+GV+9+AT0jBm7+fP35ieHvtw9Ata8YpEQVgcmKgeHbr38M6lqaDOcvXWbg4+Nn+ABMzizAGP/z6xeDrNQPhuffgEX803MM74HJ7zcLsGL+9x1ScgDzGTcXCwOwOcPw489vYJHOxcAqpsDAziMEjGFmhr+/vzN8fX2XgfnLc4Zv/0Gtf2CB5e4X9x/SCgWlyv+I3AiOSVC0Ikp5dg5OBgZ4hQeNcmhxyfX9LjDZMDJ8YFFgYEAUUQzgUgNoFiMTpHX+H9pmBJkqzPSK4e0/cQZx1lcMr/6IgfX9B2VWSAZiQFSOwDwOLL8ZwZgRLA6q+0BuAdH//0H6XEzMQM9DLGREtBBAhv3/z4AKINXrv7+/oAmWEepoSGsAJMgJTF6/fn6HOuofuKQEeew/qBgEZWxg/wYltIABwwqscBl+/Qc3if7//g/tdjAxwFsC0MoHnlv+g+pEoCy4/Qmx4/9/SICDXPjvLzAWGf7/Y0B3PgMWD4H9CXIUqKYDaUCuAIFsZlZmcFr+//c72N2wFgbCMWBNDHBPAbsQ4Ebr958MoGbQ/78/wX0nkDyiEICaAopVcMpgAuc9uHthKQEsAPEHy3+kbgM8lWF6ERozyNU21Gng+gSSyVlZQJn/N1LkwjodUCegdVFAzZr/wECCJJvfUM9Ci2VGUIIDFQ3/wbEOVgOuZIHJC5zEYNmBgQHWewVZDGzXg2p5RkTIgct9hJtgpT6okgAnH+T8AYk2kIkMT1+8Ala4PxnERf4yfGIQgFryD2IurCyG5j9ICvjH8PLte4b/LMIMr94AWxRMQmBzYPkC5gLQWAQDtPPDCBP8D8kz4PgDmQnNQ6AgZsm2kgHW+KA6BtjTBLazmEAZD9RBA/ZpQLH3B1iigTLdb2CJ9AfUvwH3c/6Cm/z/oCEFaqOxsGgyrL/zEtg/+sGQYCTIAG84As0DNXlAGJS0wCENdBhIb9LEgwxysrIMj588ZdDRVGPIddBjgGd8UAwxIrqVjKgVFAOiifQfUk6Bch6oPbq9q/Q/sA/BwAQaWwB6BGwxCyu4Ow12CLgRyQzmMwI7ZKDu7u9fPxm+fvrI8OHta4YP794B8QeGt+8+AscCfjFAcgo0LEEWgDETNH8wgTuCTLCSEkrDQh7UNATnn//QXPQfEi4weW4uNgYLKyMGSQUlaIpihAcczIOMjcEu/znYWYFNHGZgBcfOAGoNc7CzAytUTnDMcQArWE4Q5uUDtxwYgR5E9EcgBjIitePBYw5Al4FiFRTDoAYoKK+AuuQgMVDMgNt1wLYhWB6YEsBiQD5Izb+//xngakAVKDB5gfi/gA3er1+/A3u/bOAWCxO0tQ4JdEbwyBQ4dm9uWvCfAdZe+o80WAJLl4yMDCgAxIfmI5AB8DSPpg4cYv//I+okWMmJbh4oMOB9BIhN//9Dh8gYGRlg5oCS/pF9R4G9Y2CLgokRlq3AY4mMwOr/L9B8UOCxMAIrI+SMCHY+1FJINEILXpjjoJ4Gi0LFYNENdg4keMBMZLNgFTIDBmCEFvPQgglsMBOknIWW9KAYenb/EcOrt8DeMDMrAyid/WOEpAtQNfQPFItgc/8xsLx9/oyBlZ2DgQU8BAUaimKFVtCgUPoPTvuwjAouVf5D+oYQx0IHI6F55T/YKkZwCQePIZBFsNgB0cB8iqjnQGqZwOoRo3ewVAJNyCCzgQ6+c+shsAfMCOyXMYO1g5IxONyBxD9GFlCHBDymx7J6wwFIRgY1Of6D2kuMwAEPFnC3mZWFCdw+Y2cD9TiZgOLMDBzAIVpwj5MFNPABLBmBBQgbsAvNxsHGAGp6gDCIzwwqWFghYqCeKajpAxphAta+kBwHDpH/iMzNAK0+GGHVyn+4v189fsTw+MVnBiZ2LgZQwQGKPyZoxw9cwgL1fPvPzADyFnAcCegoUBsX3JwADSj9AfZX/jH8+AaMof+wyg6SF0BpFTxGzQAphcBjBEDDQJZA4hM8LAdyMzBmgRYAA58N2HNkBSZrRmY2yMgQsEkHGphhYWUDdzdAngelDtgYA4gNkmcFBhJYHBiYd6/dAXbtORm+A1veHED7fv4BuhOYx1mhngL5nAMYR/+A8sBBURaGP6Bw+g9qQ4EG7JmA0QoSAXqGkRla3kMH3MGRygAP1X//IR1D4FgOAyxl/WWEjL8xM0KHLYDm/v0LdAXQESBTv4DjB9Sp+wfM2D8geQ2cViFJ7B+0PgFXycBUIyPKyWDn4QjJN8ABGFiJCBqM+QssIUFjf+DxP9DoErDOZPn3j4kBFIr/oHkDaC8DCIN8C2r4MYH6IsCuAhMDzFO/GH7/ZwdGLzA5AsWAPQMGUFIC9p4Z2MCjrf/AYwX/oQ6DtaL+/gepB4n+B45TgEovkDpoafUf0owCxzQoxkGtG6AXWIDNIn0TEwYmVlawx4FBBVEICkgmxNg6vNIF6mVhBjuKEVjzM0BCC0gygz0BDHdGUIMPOKIJNOAXMI2CPMcMzICgqAZZDooNkCdYQen+P2Tc4TcwYFiBGR8k9hck9h8U8iAX/wMXdKBWzF/ggAcwFBhASRjktn/QqgA0RgeSAhXBTMDYERNiZ+ATEGT4D+5/MULcx8iIaLtB7YX4EhISLD+BJoBKDFCPkA3qaVBpBkwUDD+B+ZTpHwsw5EF5A5huQbEIbY2ARnzYmECe+M/wE5oIQR0xNkZIKfXzDyOwNwupTziYIDHzB2jXb6A3QAHCCKxIQZXJb2Bs/AGLAbtOQPrPX4hdoEarkZkhcEwCOGj5D1Z4QAoKRnAMAbMIqMr5D5UDlYZMoOQO7OOAkst/YND9BIYmqDvLDFQEijk2oENBIfYLkqAZQMO7oN4sqDYHqfsOShr/QaUiAwM7I6gO/QsuLv6A2oFAvWzMoMIYlC8ZQMP54KTMzABrWEL6fpBxREaGX0DP/AZ3pYHxBtQvJcrOIConywCeqQC3GICe+fMHEnSgyAKJgbocsCoBKAMqSVlAk12/QaECjGJQiLMC6f+wmhuokRmEgSECbOqBC4ifQN8Bh9PBjUhQvmJnBPGArH+QGAENd7EAQ4sdFHugQhKUr8CzEcAhLUZI8vsFTIvMkNEDYP8OOGbwH1x+MnCzAN0PHHD8A8z8RqZG4I4eaAAUkjyBAQMsKSGdRmBWAAXan/+IKgBcyQPHEVmAPUoWxn/gpgR4fJkRUtqBSjDwzNE/SBL6BQwRoCpwHgeFMiuwoGAGD+sygod/f/2B5BEOJkisAO1jYAeNs4EqPuAABxPQwn/AUvMb0BHg+gKU5EGhCjIR6AleNsjE20+gZ6QleBjEFYENUFBxD23PwduPIPcBcz6wNQAO4P9AvQygZhqoBQOkWdiBDgP7GlqOMIAsh2ZS0NQGSJiJCZIRgamZATgKDa7wQFkAlOlBoyCgOAKN0LCCSzlIsgPlJ3D7CuhAUMD8ARY0n3+AYukvcHwPFCTA6RdgLH4HJiNO8MToP/BgJsg+YzNdBlBl/B8UC//+QFoGoBgAmgXpkEK64aCYYwB3d0BpkAlcGrL8huYpkE9BxS/IL6A8AKmmGcEJAzSqCspPkFYLKGkBYw00VgBKQkAD2Rkhmf7v3//gcQFQAPwBJV1Q0gKqAdXiX4CtZTZg4QAqeIBVBniu6BcwtkAxB2oxg0aTQMWgrCQvg4iUFHCoAxjjoOYYCwcD2BMgDzFCSjpwjDBCSjVw6x6Uf4EB8x8U+2xAi/8Ak8IvUAkGHGQAz74BPQlKcqCKGDhSwAAqq1iYIbMUIM+CZqlBjVpQrICa/CAHgnMB0BLgUCzDD6AAqGIFdRZ/APPH5x+/GdiBw8IcTJAxNlDe+wlMouARcVAeBcYECwOklDOxNmHg4BcCFy4gD4IqUAZQsgKlYZAoKBkCWxmwdhGkDoJUC6B2EQDNmvC0MbRQuwAAAABJRU5ErkJggg==",
"webots_horse.png": "iVBORw0KGgoAAAANSUhEUgAAADQAAAAnCAYAAABAOmveAAARf0lEQVR4XmP0im7//58BCBgZkACQA+UzMjCiSDIyIhSC2f9RNDKAuEyMIF2MYDZMPVgVIxMD3CqQGrA1QLVMUFvAYkA21A5GqDhYD0w9I1QtE8RdYB7IHKgelv9gCSYGiLP+AfUC2ciOhroA5Gu4Z/4jLGWAexzqb6hnQEb8h1rOAPMISO1/iDqwl8FuYmRgQvYI1JNMTGgBB9IGV/efgRFH4LAwMzGBbYA4mBk9quCxxgQ1ABbEqCEPiw2IckawR6CBxIgQA4cN1KEgNUywmECiQR6BcBlhVjEwgD0CoiABCdaHEujQAAZqYWFggMQIyJ7/DECfA4P8P9QbELvB8cnA8B85xIDc/+DIZGD8D7EIltQYED6GsBiRHMaAcBQj1JEMKGJgHgPEwRA2RB5JH5J6mBxQloERGtsszMyQdA3RjpSUGBAAkjyYgH6CpxcGRiR5WEiDBA+sa2OwNDNg4JD1heQ+jFiA2gQPdUggQhzFiBIbMM+CaJgnIfkWGA1IMQ12FdRBLLDoA/H/M0JDAmgCLJYYoU4Hues/KDYZkGMK2VtgtzDwC0ozyMprMKip/Gbg5RVj2HPhAwMiJCGhAMnPoHzACM4LsNhCzuDI9kKSHBM47cCSIyNSFkCIAd0QlDbxP9wgeKgzMjCipX3k0GJAiR+IWpD6i4dXMvz/9xus9+ePrwz2NuYMjva2DNvPfIVmfEjsMMACjhEcLwwQD6HaCS+AkAIZ7DGoGxmxxjwTA6RQYIRECcxgsIUMUMsZICHJgMWzDEjg4MaJDKKiogw/f/5i+P71IwPIqcdOnmcQEOAHWqICDQMmsGdB1jFCHQoygglWBDNCHAJ3B1LqQOQ5SEgzQgODAYkGFWxMzMzMDMxMzAysrCwMLCxANjBPgUo+JmDpB8pfEHkmBmaoOAuUZkaWB5rhFFTE8ODODQYTl1QGW78Shr9//zD8+/ubYde+4wxmyn8YIOoZgTSwmGYGYSYGJmaIuSAPMYPEmCB5A8RnArMZIPYC1YH5jCD9DGB9jExM4OTKxMjMANbPBI4dBsaY/Bn/USpLUGQxQnIQOFTAEQUMBkR5AIt0cCwwQNUy/IemeiZo8QeUPbp5EgMrGw+DhroyQ0xkMMPWE+8YQJUlLMXCKmB4HsJIRojYYGBAZkPyOjjBIsUqAzCKWJhZwGUYRP1/SIkDEoEXEFCPgNIKSuXKAE/MKDmKAZx1IUnrz58/wOD8zXDp8g2Go4qnGWx1tBgkJCTAsXf9xi0GURFhoBv+MVy48pjhD6cM1LOQwIMkS3RPQOpMRqjViKQKEQC5j4UVmMwY/sP9Dy8MGKAxA9PMAAfQSpSBgYEBU5IBkVkZGH7/AhYQTH+ASYSNYduOAwyvXr1m0NbSACYRZoYfP34wtMxezCAjKczg7+cF5H9gePlbGGELI2oVAjMXFKvgNMDEBHMiLGTBlSMLCwtS+wrJ0VD/IBQzMDDAMyIDJNpgdS02T4MUs7CyAXX9A3rsK8PfP0wMJ0+dYzh34RowJpgZ/vz+yfD5wyuG298+M+w/cICBi1eGQVpDlAGShBnhles/oKVMDP8YHt24yPDpzROGH9++AeX+M8ioGTFIKKszsDCygMP1N7DVxsr4lwEYQ5jOgVV14FYENPpAIfT/P7TEY4QmUwYkgGQM439IlP//9xOoGxhDjOwM1pZGDIePnAAWFL8YQG27P39+AfMXKziQzpy9zODirgh0DCO4rQKJhf8M//79Z/jw8DrD++fXwbENjHBgwcXIAIqDx3dOMLy4d5xBUESD4cevTww/P78F6v0NNANYQv1nACYNoD9//PjG8ODaWYavX78xcLP+Y+DjYmf4x8jKwM4jwMDGwc/AISTCwMUlAE5qsLbd3///GGDJAJo/GWA1NysbF8PnT28ZePg4GHbu3M7AxsYB9NBPcCj8+vUL6DhgycrKBI4VPj4eYFL8B9TLzMDG+Ifh7IGNDPxcTAzfvnwDe/I/MDlwANV++f6XgY0TaMQfFobX778wfP56HZzUfv3+w/D9N9BDD8/sZPgFDEk2Fg6Gv/9/MTD9YWTgYP7D8PsvA8O7Lz8ZmBl/MHz4+B5o2X8GticsDJwsbAxi8uoMvBJqwIINaDiwuP37H5REILH1HxZpQN/9/fuXgZ2DF+iJvwzppb3ASvcvMNT/MLACzZjVV8nwG5hOmJh+A3WwMHBw8jBcPX2M4f7dWwzMwMAVE+ZgePXtHzidgwoXUJXCyPiP4eevP+DM++/vP4Y/wCj7/AfUfvnL8BfUUQC2HhhdPSP+ozdKORi/Qp0FLmvAbJA2SAkGE0OkMWghD1H3HxgSDLCim5HhyrVLDEAfMeho60LMZITp+89w/vxpBkMDE4bzF88zWJmbM/wAVsqwYPn/H168MsADiQHJPTB5oCf+w0snYMACUyowykBRDfQiyDH/QaH4DxiS/8BF6j8Q/z+I/RfIBzr2PzCEUDCyGDC0wXIgMRD7J7AwAMY2ML/8B3v0H6SZDnLMfwaw+XLyUsCY+4PkGQaklj3IKzAM8gwynwFc4AA1A8sRkH0QO1kYwBaBDGGAA0iMQbsIcClEOMEUoojAMg4DTB9EFlTS/f8H6meBAoQJ4gioAYYGxgwPHz4C1/j///1FVAMgz8LjBRorjKDAYEB1KNicvwzILmcBlSuQ6EUII1oOYFkGRiRj/sMsgvoGzEdkHAZ0wMHJDQyz3wxgOxj/IUIJqlBBXhYs9p8BFMIgQUiShJeoDFBP/GdkwAxSiCH/kSSYwFEGKUeAsv/ATgc58h84Z/2HGgejYRYywqxBuJ8RYjF64Lg52wAzLDTf/IeYz8AASzoMDA8ePGa4dOUGMNBAsYfUN4PnNQYkN/yHBgiU/g/JFgjz/jOwhKkIQmpYVmZwLQ9qjEIah8CGIKjhCsLA1gSwFQtOGiC3gDz7588/cKiD6gpYWgE1NkHqwEU61A8g+U2/vzNk2OhB7AUZAHQsyOPZM9cxTE0LZMi9foch1UobKfb+Q1ssSNkdah6sPoR4E7U1ASpVGWsCgv6DY+M/xIHgOECOW0ZIrPyHliaMwJY5qGIEGwxO3swMDIyI9iCo4yXAzcogLyPMoKAsz8AnJMgA0QN079+/4IKAARzAQMdCmy+QigvS4GSENEcYYEkO1KpmggbAP1ALHtg+/P/vHwO4tQ3U/x3YcngOzIePHz5jeP7qEwPjkRk9QL3/wHUFA9RkmH9ASQ91jAuSp0Ap5x/Q0D9//jL8+P6T4R2wgnv/8RvDH2BlIMTPycDLzcHAzMIMiVFGUPwxwpMmpGXNCG7yM8K60QwwzzBAmleMsP7qf3Dkg/QgBk+QPQ4yl4kB0nAGRgjQTSwicvLwdAmMKgYGWNaDxgzMARBNQFlgc+Tb5y8Mr1++Znj28h3Dw+cfGH6ASmiwI5gY3v0AtgQYfzKgZS4GRlg0MMASCwPEsVBxiJf/I5Llf0gB8h9Mg9gQB4PzPDgWmcA5HzxEBgsvoHYWSGeRCZINmKElCTBa/oGSBxD/BjZRvn76xPDx/SeGt28/Mrx885nhzedfDD/BpSwkdMBOBJZC/yBuhAYQwhZY/oaVU9CwYmCA5XFoICLUgVRCmkSg/AjxFMhwaL6Fp1GQGpgbIPaxgJIXyPEgR/0Dtof+Az0Bamp8eP2a4cXTlwxPnr1jeA/0ACM4D8GzIgMnPBUxIsKckQFel/+HF/ZQb8B9AXIYEwOkboUkYUgJwAj2IEgZrB6EJFZoHw0kDeydQspHRkgyA9sMYTNAy26WX1+/g1uyH16/YXj7+h3D6zcfGF69+8oAajIxwjUwM8AKXAZYzxFoMmQsjxEsB4kl9OTEiBD+DzcNKAaJSyZgbxfWggHn2//wWg4SZ6AYBPsT6PH/iDoRFmwQzzEwQNroIKuALfE9u08AW63fgPngH8ST/yEhCKsRoIUONBYwij9IsmFAAoyQcIX1zMGWMTLAW1tgR/yHOgkcuEDiH7j4gXgeyQpw6fYPIgDzPChYIEkTIv4P6ivIyBWwHrIw14TWJ0APAUupv+Ak9xfcBgMWjsDuMrBVC2yW/wW3bv8Axf+CY+QvqHvNyARhg+SAJR7I8P/QCvnf3//AUu8vuE8DSjSg+giGwWqADgHzQTQotqH5CBJTDJCqHqkJAE6G0NLvPyRdwnX8Z2CExyiLiIwMw/8/wGLqHyTDgUf8WdmBzZU/DH9+/mAAlXxMwLrn339I2f8f1E4HepqFjZXhHxOkDgLlOyZWVkgxDRqNgbYI/gPT/F9gocIIlIPkY6BL/oIat9DUAHTgZ2BpycrOxsDKwwWuWxhA1QFIDxvQDUD619cvwIYrsMsBtApk3+8fv8CjPj+BNDDMGP6CAhsYuH/BocnAwPLvx3cGRlCdwQIZqP8H9Ny/b5DuAyMz0JHA0W9QxcgErtD+MjCzAy0CGvKPmQXcvwF5hhHJE8AoZABXwqCgBvazwPx/QE8AY5OJhYWBCWjP31/ATiGwew6KKRaQGGgICiwHMvMfeKgLFCbMgsLADiUHMKb/A3ukX4CdTFYGJmFW4NgWsGPIxsYAKoFB9oOrG2jjlgnY22IAddD+AEPj76+f4LTOwgGsGNk5GEChDi71vn1hAGcWYMyAQpCZi5vhz/fvDH+Aofcf2DUApfV/4GILmDCAsQLqfP0HxsQfoHngZAVKxt+/QfMbsBABiv8HjtmBUgZIHUgClIRBLQFQiQcObKD8388fGP4DUwEorNm4ORl+fvsOqggZ/v/8zvDv9y9gtwFYuIBaDcAeAyiGQY5n+QscfQGlQGZQTmMBJiNQMwPUxADFAhAzAT0GLrJB3VtuHoa/wGT4/cUzcK5mAVrCDEqeQM8wAuuwf8BY/AfsxoNi4t8/SHcBFCMgRzKBPAwsKUD6GYGxAUrav4AOBHke3E4EhTCw98nIxgxuZYD6T3+B7UVGoJmgFhJoToGNi5MBlMdBAQ2KHXBLgxnUnmNh+Aus0Fm5eIBjCsCBwf9MwHQFSgLAoPkHHI2BJSNg+4ThNzAm2Hj5GEAjqT8/vGf49+sHAxMwupk5ORkYQXqAoQ9yEDipAkMOHMJAh4FCmREYWyBL/wHFwW00UIUNDBCQJ4FD4Ay/gGMXjMDQZWaBNV8YwDEAqS0gOf/vz58MTDzA8QagICswmYNHZIHZggU4zvEXOH7OAvItMCJARTSY/5+DBxx9f4HJB+QBULSBGn9/gd1hUH7hBI5N/wXmsx9fvoAblkzADMzCCczAwHEBUKeMCTTwAfTkfyAG6YdUmEAHApMCEwczA1gcKAiKaZC5IMeCKvLfb9+CUi8DOyc7OA+Dkh6kUGMFV5qMwFY+AyiJgxu0wHIMWEiA+cDkChwDY/j9+T24RfIbmKJYQWqBBjMDCx+WPz9A6RGYd0CZHFRigPIRMGTZBQTAzaRfn94zgPIXyAIWdm4GFg7gkMvff+BOGyPQU6BQ+f8boh+croEFyX9gMmEGxuL/P8BhLFAeA+oB6f/z5SskLwJJJg52BlagfaAA/AuOWeBoK2gE6h+k+w8qaEA+BpkJSsaMQDPASRlUgP2DNNVA+Q4cj8AswgKMpV/A7MMCGlAHJYe/wIwPKi1AyYuZi4sBVNz++vQZPMABSiLMXLzAtM0KyYyg4poJlG6/gJMIMzCf/QbGECO4uP8HLlZBGR3sEGDIMgLV/gNXAX8YGEF88IA/MOdCJ4WZOLjB1cR/YGkIrmMgjQoGUF77Dyxxf336wACKBVZg3gMF6F9W4HDYt08M/6GzFqDk/Q9YioLz0H9gafEPOBzEAgoxYF75B7T856d3kBgA1ejAmAOlV1By+vv9C7SuADr3zw+gZ/8AY4wLWPoAPQ50CTMwOf4DdrOZ2LnAdRsTKGaAngeXeMBSDlxXgTwEGngBVkzMbKDkBZw9ADoUNJIKn1EA8sFjHSys4OT47xukRAMlVdBIL8jjTOycwCG3XwygWAEVNMzAYTCm/38YAIEv8QtBH51oAAAAAElFTkSuQmCC"
}
print(f"✔ {len(SAVED_IMAGES)} saved images available: {', '.join(SAVED_IMAGES)}")

In [ ]:
#@title 6.2 Choose and classify the images
IMAGE_SOURCE = "upload"  #@param ["upload", "my saved images"]
SAMPLES = [("car.jpg", "My photo (car)"), ("cat.jpg", "My photo (cat)"), ("bird.jpg", "My photo (bird)"),
           ("webots_dog.png", "Webots e-puck camera (dog)"), ("webots_deer.png", "Webots e-puck camera (deer)"),
           ("webots_horse.png", "Webots e-puck camera (horse)")]
IMAGE_TYPES = (".png", ".jpg", ".jpeg", ".bmp", ".webp")

def choose_images(folder="my_images"):
    """Ask the user to upload images. Returns [] if cancelled or not in Colab."""
    try:
        from google.colab import files
    except ImportError:
        return []
    os.makedirs(folder, exist_ok=True)
    info("Click 'Choose Files' to pick your images, or 'Cancel upload' to use the saved images")
    try:
        uploaded = files.upload()
    except Exception:                       # upload cancelled
        uploaded = {}
    items = []
    for name, data in uploaded.items():
        if not name.lower().endswith(IMAGE_TYPES):
            info(f"{name}: not an image, skipped"); continue
        path = os.path.join(folder, name)
        with open(path, "wb") as f:
            f.write(data)
        items.append((path, f"Image {len(items) + 1}"))
    return items

def sample_images(folder="saved_images"):
    """Write the saved images (embedded in this notebook, cell 6.1) to files."""
    import base64
    os.makedirs(folder, exist_ok=True)
    items = []
    for name, label in SAMPLES:
        path = os.path.join(folder, name)
        with open(path, "wb") as f:
            f.write(base64.b64decode(SAVED_IMAGES[name]))
        items.append((path, label))
    return items

def predict(path):
    """Return (RGB image, 32x32 model input, class probabilities)."""
    rgb = np.array(Image.open(path).convert("RGB"))
    x = preprocess(torch.from_numpy(np.ascontiguousarray(rgb)))
    model.eval()
    with torch.inference_mode():
        prob = F.softmax(model(x.to(DEVICE)), 1)[0].cpu().numpy()
    small = (x[0] * torch.tensor(CIFAR_STD).view(3, 1, 1) + torch.tensor(CIFAR_MEAN).view(3, 1, 1)).clamp(0, 1)
    return rgb, small.permute(1, 2, 0).numpy(), prob

header("Step 6.2  Choosing the images")
items = choose_images() if IMAGE_SOURCE == "upload" else []
if items:
    info(f"{len(items)} image(s) uploaded")
else:
    info("Using the saved images: my car, cat and bird photos and the Webots e-puck camera images (dog, deer, horse)")
    items = sample_images()

header(f"Step 6.3  Classifying {len(items)} image(s)")

fig, axes = plt.subplots(len(items), 3, figsize=(10, 3.1 * len(items)),
                         gridspec_kw={"width_ratios": [1.4, 1, 1.6]}, squeeze=False)
IMAGE_RESULTS = []
for row, (path, label) in zip(axes, items):
    rgb, small, prob = predict(path)
    top = prob.argsort()[::-1][:3]
    info(f"{os.path.basename(path)} ({rgb.shape[1]} × {rgb.shape[0]} px) → {CLASSES[top[0]]} ({prob[top[0]]*100:.1f}%)")
    row[0].imshow(rgb); row[0].set_title(f"{label}\n{os.path.basename(path)}", fontsize=9); row[0].axis("off")
    row[1].imshow(small, interpolation="nearest"); row[1].set_title("what the CNN sees (32 × 32)", fontsize=9); row[1].axis("off")
    row[2].barh([CLASSES[t] for t in top][::-1], prob[top][::-1] * 100, color=[BLUE, BLUE, NAVY])
    row[2].set_xlim(0, 100); row[2].set_xlabel("probability (%)")
    row[2].set_title(f"Prediction: {CLASSES[top[0]]} ({prob[top[0]]*100:.1f}%)", fontsize=10, fontweight="bold")
    IMAGE_RESULTS.append({"Image": os.path.basename(path), "Source": label, "Prediction": CLASSES[top[0]],
                          "Confidence (%)": round(float(prob[top[0]]) * 100, 1),
                          "2nd choice": f"{CLASSES[top[1]]} ({prob[top[1]]*100:.1f}%)"})
plt.tight_layout(); plt.savefig(f"{OUT}/image_predictions.png", bbox_inches="tight"); plt.show()
show_table(pd.DataFrame(IMAGE_RESULTS), "Predictions on the chosen images", fmt={"Confidence (%)": "{:.1f}"})
print(f"✔ Saved {OUT}/image_predictions.png")

**Notes on the Webots images.** The e-puck camera gives only 52 × 39 pixels, the animals in Webots are 3D models (not photos like CIFAR-10), and the camera sits about 3 cm above the floor, so the robot sees the animal from a low side view. Because of this *domain gap* the CNN is less confident on the Webots images than on real photos, and it can confuse similar four-legged animals (dog, deer and horse). Fine-tuning on a few hundred labelled Webots images would close this gap.

---
## 7. Optimise for the robot

The e-puck updates its motors every **64 ms**. If the CNN takes too long, the robot moves late. Four versions of the model are compared on the **CPU with one thread**, like a small robot computer:

| Version | What changes |
|---|---|
| **FP32** | The trained model as it is (32-bit numbers) |
| **Dynamic int8** | Dense-layer weights stored as 8-bit integers |
| **Static int8** | Weights *and* activations in 8-bit, calibrated on 2,000 images |
| **Pruned 50 %** | The smallest half of the weights set to zero, then fine-tuned for 3 epochs |

In [ ]:
from torch.ao.quantization import quantize_dynamic, get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import torch.nn.utils.prune as prune

engines = torch.backends.quantized.supported_engines
torch.backends.quantized.engine = "x86" if "x86" in engines else ("fbgemm" if "fbgemm" in engines else "qnnpack")
cpu = torch.device("cpu")
fp32 = copy.deepcopy(model).to(cpu).eval()

def size_mb(net):
    buf = io.BytesIO(); torch.save(net.state_dict(), buf); return buf.tell() / 1e6

def latency_ms(net, runs=300):
    torch.set_num_threads(1); x = torch.randn(1, 3, 32, 32); net.eval()
    with torch.inference_mode():
        for _ in range(20): net(x)
        t = time.perf_counter()
        for _ in range(runs): net(x)
    torch.set_num_threads(os.cpu_count())
    return (time.perf_counter() - t) / runs * 1000

header(f"Step 7.1  Building the optimised versions (quantisation engine: {torch.backends.quantized.engine})")
dyn = quantize_dynamic(copy.deepcopy(fp32), {nn.Linear}, dtype=torch.qint8)
info("Dynamic int8: done")

calib = GPULoader(X_train, y_train, 100, shuffle=True)
prepared = prepare_fx(copy.deepcopy(fp32), get_default_qconfig_mapping(torch.backends.quantized.engine),
                      example_inputs=(torch.randn(1, 3, 32, 32),))
with torch.inference_mode():
    for k, (xb, _) in enumerate(calib):
        prepared(xb.cpu())
        if k == 19: break                                  # 2,000 calibration images
static = convert_fx(prepared)
info("Static int8: calibrated on 2,000 training images, done")

pruned = copy.deepcopy(model).to(DEVICE)
targets = [(m, "weight") for m in pruned.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
prune.global_unstructured(targets, pruning_method=prune.L1Unstructured, amount=0.5)
ft_opt = torch.optim.SGD(pruned.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
zeros = sum((m.weight == 0).sum().item() for m, _ in targets) / sum(m.weight.numel() for m, _ in targets)
info(f"Pruning: {zeros*100:.0f}% of the weights set to zero, fine-tuning for 3 epochs...")
for e in range(3):
    bar = tqdm(total=len(train_loader), unit="step",
               bar_format=f"   fine-tune {e+1}/3  {{n_fmt}}/{{total_fmt}} {{bar}} {{elapsed}} - {{desc}}")
    train_epoch(pruned, ft_opt, bar=bar); bar.close()
for m, n in targets: prune.remove(m, n)
pruned = pruned.to(cpu).eval()
info("Pruned 50 %: done")

header("Step 7.2  Measuring accuracy, size and speed on the CPU with one thread (about 1 minute)")
OPT_RESULTS = []
for name, net in [("FP32", fp32), ("Dynamic int8", dyn), ("Static int8", static), ("Pruned 50 %", pruned)]:
    _, acc = evaluate(net, device=cpu)
    ms = latency_ms(net)
    info(f"{name:<13} accuracy {acc*100:5.2f}%   size {size_mb(net):5.2f} MB   {ms:5.2f} ms per image")
    OPT_RESULTS.append({"Model": name, "Accuracy (%)": round(acc * 100, 2), "Size (MB)": round(size_mb(net), 2),
                        "Time per image (ms)": round(ms, 2), "Images per second": round(1000 / ms),
                        "Share of a 64 ms step (%)": round(ms / 64 * 100, 1)})
opt_df = pd.DataFrame(OPT_RESULTS)
base = opt_df.iloc[0]
opt_df["Speed-up"] = (base["Time per image (ms)"] / opt_df["Time per image (ms)"]).round(2).astype(str) + "×"
opt_df["Smaller by"] = (base["Size (MB)"] / opt_df["Size (MB)"]).round(1).astype(str) + "×"
show_table(opt_df, "Optimised versions on the CPU (one thread); green = best",
           fmt={"Accuracy (%)": "{:.2f}", "Size (MB)": "{:.2f}", "Time per image (ms)": "{:.2f}",
                "Images per second": "{:,.0f}", "Share of a 64 ms step (%)": "{:.1f}"},
           highlight=["Time per image (ms)", "Size (MB)"])
print("✔ Optimisation comparison finished")

In [ ]:
header("Step 7.3  Comparing the four versions")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
names = opt_df["Model"].tolist(); colours = [NAVY, BLUE, GREEN, AMBER]
for a, col, title in zip(ax, ["Accuracy (%)", "Time per image (ms)", "Size (MB)"],
                         ["Accuracy (higher is better)", "Time per image (lower is better)", "Model size (lower is better)"]):
    a.bar(names, opt_df[col], color=colours)
    a.set_title(title, fontsize=11); a.tick_params(axis="x", labelsize=8, rotation=15)
    lo, hi = opt_df[col].min(), opt_df[col].max()
    if col.startswith("Acc"):
        a.set_ylim(max(0, lo - 5), min(100, hi + 3))   # zoom in: the differences are small
    else:
        a.set_ylim(0, hi * 1.25)
    for k, v in enumerate(opt_df[col]):
        a.text(k, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/optimisation_comparison.png", bbox_inches="tight"); plt.show()
print(f"✔ Saved {OUT}/optimisation_comparison.png")

**Running the CNN less often.** The robot does not need a new label every 64 ms. Classifying one frame every *N* steps divides the CPU load by *N*:

In [ ]:
header("Step 7.4  CPU load when the CNN runs only every N control steps")
fast = opt_df.loc[opt_df["Time per image (ms)"].idxmin()]
show_table(pd.DataFrame([{"Classify every": f"{n} step{'s' if n > 1 else ''}", "Interval (ms)": n * 64,
                       "CPU load, FP32 (%)": round(base["Time per image (ms)"] / (n * 64) * 100, 2),
                       f"CPU load, {fast['Model']} (%)": round(fast["Time per image (ms)"] / (n * 64) * 100, 2)}
                      for n in (1, 4, 8, 16)]),
           "CPU load when the CNN runs every N control steps",
           fmt={"CPU load, FP32 (%)": "{:.2f}", f"CPU load, {fast['Model']} (%)": "{:.2f}"})
print("✔ Proposal for the robot: static int8 model + classify every 8 steps + run the CNN in a background thread")
info("The Webots controller (Assignment_3.py) uses the background thread and one CPU thread; set USE_INT8 = True for int8")

---
## 8. Save and download

* `cifar10_cnn.pth`: copy it into `controllers/Assignment_3/` in Webots for live predictions.
* `cnn_outputs.zip`: every figure and table from this notebook, for the slides and the report.

In [ ]:
header("Step 8.1  Saving the model, figures and results")
torch.save(fp32.state_dict(), "cifar10_cnn.pth")
info(f"cifar10_cnn.pth ({os.path.getsize('cifar10_cnn.pth')/1e6:.2f} MB): copy into controllers/Assignment_3/ in Webots")
results = {"test_accuracy_%": round(TEST_ACC * 100, 2), "parameters": TOTAL_PARAMS, "epochs": EPOCHS,
           "training_minutes": round(TRAIN_TIME / 60, 1),
           "per_class_accuracy_%": {c: round(float(a) * 100, 1) for c, a in zip(CLASSES, per_class)},
           "image_predictions": IMAGE_RESULTS, "optimisation": OPT_RESULTS}
json.dump(results, open(f"{OUT}/results.json", "w"), indent=2)
opt_df.to_csv(f"{OUT}/optimisation.csv", index=False)
!cp cifar10_cnn.pth cnn_model.py {OUT}/ && zip -qr cnn_outputs.zip {OUT}
for f in sorted(os.listdir(OUT)):
    info(f"{OUT}/{f}")
try:
    from google.colab import files
    files.download("cifar10_cnn.pth"); files.download("cnn_outputs.zip")
    print("✔ Downloading cifar10_cnn.pth and cnn_outputs.zip to your computer")
except ImportError:
    print("✔ Saved cifar10_cnn.pth and cnn_outputs.zip")

---
## 9. Summary, key findings and future work

In [ ]:
header("Step 9.1  Summary of the results")
summary_rows = [("Test accuracy", f"{TEST_ACC*100:.2f}% on {len(test_set):,} test images"),
                ("Model", f"{TOTAL_PARAMS:,} parameters, {EPOCHS} epochs, {TRAIN_TIME/60:.1f} min on {DEVICE.type.upper()}")]
summary_rows += [(r["Source"], f"{r['Prediction']} ({r['Confidence (%)']}%)") for r in IMAGE_RESULTS]
s8 = opt_df[opt_df.Model == "Static int8"].iloc[0]
summary_rows.append(("Best for the robot", f"Static int8: {s8['Accuracy (%)']}% accuracy, {s8['Time per image (ms)']} ms, "
                     f"{s8['Size (MB)']} MB ({s8['Speed-up']} faster, {s8['Smaller by']} smaller)"))
show_table(pd.DataFrame(summary_rows, columns=["Item", "Result"]), "Summary of the results")
print("✔ All steps finished")

### Key findings
* A compact VGG-style CNN (about 0.8 M parameters) reaches **about 88% test accuracy** on CIFAR-10 in a few minutes of GPU training, far better than a simple 3-layer CNN (about 70%).
* **Vehicles are recognised best**; **cat and dog** are confused most often because they look alike at 32 × 32 pixels.
* My own photos are classified correctly when the object fills the picture; the **Webots camera image** is harder (low resolution, rendered 3D model, low viewpoint).
* **Static int8 quantisation** makes the model about 4 times smaller and about 2 times faster on the CPU with almost no loss of accuracy, so the CNN fits easily inside the robot's 64 ms control step.

### Future work
1. **Fine-tune on Webots images**: collect labelled e-puck camera images of the animals and blocks, and fine-tune the CNN to close the gap between photos and simulation.
2. **Detect and classify blocks in Webots**: add the coloured blocks as extra classes, or use a small detector (for example SSD-MobileNet or YOLO-nano) to find *where* the object is in the image, so the robot can steer towards it.
3. **Lighter models**: structured pruning (removing whole filters) and a MobileNet-style network for real speed-ups on an embedded processor.
4. **Edge deployment**: export the int8 model to ONNX or TensorFlow Lite and run it on a microcontroller-class board on a real e-puck.

### References
* A. Krizhevsky, *Learning Multiple Layers of Features from Tiny Images*, University of Toronto, 2009 (CIFAR-10).
* K. Simonyan and A. Zisserman, *Very Deep Convolutional Networks for Large-Scale Image Recognition*, ICLR, 2015 (VGG).
* K. He, X. Zhang, S. Ren and J. Sun, *Deep Residual Learning for Image Recognition*, CVPR, 2016 (ResNet).
* S. Ioffe and C. Szegedy, *Batch Normalization*, ICML, 2015.
* B. Jacob et al., *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference*, CVPR, 2018.
* Cyberbotics, *Webots R2023b documentation: e-puck*, https://cyberbotics.com/doc/guide/epuck